[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Files, Paths and Formats](https://johnfisher-ai.github.io/Python-Visual-Guides/files-paths-and-formats.html)

# A Small Pipeline


## What you will be able to do

Take an archive of data files that were produced by different people on different machines, read
every one of them, keep the rows that can be used, and write a single clean result alongside an
account of everything that was dropped and why.


## The idea

### The problem

A colleague sends you a zip file of last March's temperature readings, collected by four field
teams. Inside are four CSV files and a text file of notes. Each team used whatever was on the
machine in front of them.

One team exported from Excel, so their file begins with a byte order mark. One saved on an older
Windows machine, so their file is latin-1 and the station name reads as bytes Python cannot
decode as UTF-8. One file has a row where the reading was never typed, and another row where
somebody wrote `warm` instead of a number. The notes file is not data at all and has no business
in a CSV reader.

Every notebook in this guide has handled one of those problems on its own, with a file built to
have exactly that problem and nothing else. Real work does not arrive that way. It arrives as one
archive, with several problems mixed together, and nothing on the outside saying which file has
which.

The instinct is to write one loop: open the archive, and for each member decode it, split it,
check the fields, add up the totals, and append to an output file, all in one pass. That loop
works on the first archive. Then a fifth file arrives with its temperature column named `temp`
instead of `celsius`, and there is nowhere in the loop to put that check, because every part of
it is entangled with every other part.

### What a pipeline is

> A **pipeline** is a sequence of stages, each with a single job, in which the output of one stage
> is the input to the next. Any stage that discards data also records why, so that the number of
> rows that came out can always be reconciled against the number that went in.

### Why it is built this way

The first half of that definition is about testing. When decoding is its own function, you can
hand it a byte string and check what comes back, without an archive, without a CSV, and without
anything on disk. When it is four lines in the middle of a loop, the only way to test it is to
run the whole job.

The second half is the part people skip, and it is the more important one. A pipeline that drops
a row without saying so produces a result that looks complete. Six rows come out, and nothing
anywhere records that eight went in. A month later, when a total looks low, there is no way to
tell whether the data was missing at the source or thrown away by your own code.

So the rule is that rows kept plus rows dropped equals rows read, and the pipeline reports all
three. A malformed row is not a bug in your program. It is a fact about the input, and facts
about the input belong in the output.

Stage order is forced by what each stage needs. You cannot parse fields before you have text, and
you cannot get text before you have decided how to decode the bytes. Each stage depends on the
one before it and on nothing else.

### Where you will meet this

Any job that runs on a schedule against files somebody else produces: nightly exports, survey
downloads, log files, instrument output, monthly returns from four regional offices. The
**Pandas** guide replaces stages two and three with a single call to `read_csv`, which is faster
to write and makes its own decisions about bad rows on your behalf. Knowing the stages is how you
know which decisions those were.

### What this notebook covers

Five stages, run against one archive: list what is inside without extracting it, decode each
member and record which encoding worked, parse and check every row, summarize what happened, and
write the result safely. Then all five as a single function you can run twice.

### A first look

Here is the finished pipeline, called once. There is nothing to run yet: read it, and read the
output underneath it.

```python
report = run(archive, out_dir)

print(report["rows_kept"], "rows kept")
print(report["rows_dropped"], "dropped:", report["reasons"])
```

```
6 rows kept
2 dropped: {'missing celsius': 1, 'celsius is not a number': 1}
```

Two numbers and a breakdown. The second line is what separates a pipeline from a script that
happens to produce a file.


## Setup

Nine imports, a folder to work in, and the archive itself.

- `csv` reads each file's rows as dictionaries and writes the cleaned result
- `io` wraps decoded text so that `csv` can read it as though it were a file
- `json` writes the report
- `os` provides `os.replace`, the single operation that puts a finished file in place
- `shutil` removes the scratch folder at the end
- `tempfile` creates the temporary file every result is written to first
- `zipfile` reads members out of the archive without extracting them
- `Counter` tallies drop reasons and encodings for the report
- `Path` builds paths and reads the finished files back

The `SOURCES` dictionary below is the messy input, written on purpose: each entry pairs an
encoding with the text to store under that name. Normally somebody else would hand you the
archive and you would not know any of this. Here you do, which means you can check the pipeline's
report against what actually went in.

**Run this cell before the rest of the notebook.**


In [1]:

import csv
import io
import json
import os
import shutil
import tempfile
import zipfile
from collections import Counter
from pathlib import Path

scratch = Path("scratch")
if scratch.exists():
    shutil.rmtree(scratch)
scratch.mkdir()

archive = scratch / "readings.zip"
SOURCES = {
    "2026/north.csv": ("utf-8",
        "date,station,celsius\n2026-03-01,Troms\u00f8,-4.1\n2026-03-02,Bod\u00f8,-2.6\n"),
    "2026/south.csv": ("latin-1",
        "date,station,celsius\n2026-03-01,M\u00e1laga,18.9\n2026-03-02,M\u00e1laga,19.4\n"),
    "2026/east.csv": ("utf-8",
        "date,station,celsius\n2026-03-01,Krak\u00f3w,7.2\n2026-03-02,Krak\u00f3w\n"
        "2026-03-03,Krak\u00f3w,warm\n"),
    "2026/west.csv": ("utf-8-sig",
        "date,station,celsius\n2026-03-01,Galway,11.5\n"),
    "notes.txt": ("utf-8", "March readings, collected by four teams.\n"),
}

with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for name, (encoding, text) in SOURCES.items():
        z.writestr(name, text.encode(encoding))

print("archive:", archive.name, f"({archive.stat().st_size} bytes)")


archive: readings.zip (793 bytes)


## Worked examples

### The five stages

Each stage is one function, small enough to read on a single screen, with a job you could
describe in one sentence.

| Stage | Job | Built on |
|---|---|---|
| 1. Manifest | Say what is in the archive, and skip what is not data | **Compression and Archives**, **Directories** |
| 2. Decode | Turn each member's bytes into text, and record how | **Encodings** |
| 3. Clean | Parse rows, keep the usable ones, record the rest | **CSV** |
| 4. Summarize | Count, group, and total | |
| 5. Write | Put the result and the report on disk | **Writing Safely**, **JSON on Disk** |

The stages are built in that order below, and each is run on its own before the next one is
written. That is worth doing in your own work too: a stage you have watched produce correct
output is a stage you can stop thinking about.


### Stage 1: what is in the archive

`ZipFile.infolist` returns one entry per member, each carrying the name, the original size and
the stored size. Reading it does not extract anything and does not decompress anything, so this
is cheap even on an archive of several gigabytes.

The filter does two jobs: it skips directory entries, and it skips `notes.txt`, which is a real
file and simply not data. A pipeline decides what it will read, rather than reading whatever it
is given and failing later.


In [2]:

def manifest(archive_path):
    """List the CSV members of an archive, without extracting anything."""
    entries = []
    with zipfile.ZipFile(archive_path) as z:
        for info in sorted(z.infolist(), key=lambda i: i.filename):
            if info.is_dir() or not info.filename.lower().endswith(".csv"):
                continue
            entries.append({"name": info.filename,
                            "bytes": info.file_size,
                            "stored": info.compress_size})
    return entries


found = manifest(archive)
for entry in found:
    print(f"  {entry['name']:<16} {entry['bytes']:>3} bytes, stored as {entry['stored']:>3}")
print(f"  {len(found)} of {len(SOURCES)} members are CSV")


  2026/east.csv     87 bytes, stored as  60
  2026/north.csv    67 bytes, stored as  59
  2026/south.csv    67 bytes, stored as  54
  2026/west.csv     47 bytes, stored as  49
  4 of 5 members are CSV


`2026/west.csv` is stored as more bytes than it started with. Compression adds a small fixed
overhead per member, and on a file this short the overhead is larger than the saving. It is not
an error, and it is why the **Compression and Archives** notebook measured a ratio on a file
worth compressing rather than on a two-line one.


### Stage 2: decode, and record which encoding worked

The archive holds bytes. Every member has to be decoded before anything can parse it, and the
teams did not agree on an encoding.

The attempt order is the whole design of this stage:

- `utf-8-sig` decodes UTF-8, and removes a byte order mark if one is present. It covers both the
  ordinary UTF-8 files and the Excel export.
- `latin-1` decodes any sequence of bytes at all. The **Encodings** notebook showed why that
  makes it dangerous: it never raises, so whatever is left over reaches it and comes back looking
  like text.

Because `latin-1` accepts everything, it has to be last. Put it first and every file decodes
successfully, which is a different thing from correctly.

The function returns the encoding as well as the text, and the pipeline keeps it. If a report
says three files were UTF-8 and one was latin-1, that matches four teams with four laptops. If it
says all four were latin-1, something is wrong with the order.


In [3]:

ATTEMPTS = ("utf-8-sig", "latin-1")


def decode(raw):
    """Return the text, and the name of the encoding that produced it."""
    for encoding in ATTEMPTS:
        try:
            return raw.decode(encoding), encoding
        except UnicodeDecodeError:
            continue
    raise ValueError(f"none of {ATTEMPTS} could decode these bytes")


with zipfile.ZipFile(archive) as z:
    for entry in found:
        text, entry["encoding"] = decode(z.read(entry["name"]))
        print(f"  {entry['name']:<16} {entry['encoding']:<10} {text.splitlines()[1]}")


  2026/east.csv    utf-8-sig  2026-03-01,Kraków,7.2
  2026/north.csv   utf-8-sig  2026-03-01,Tromsø,-4.1
  2026/south.csv   latin-1    2026-03-01,Málaga,18.9
  2026/west.csv    utf-8-sig  2026-03-01,Galway,11.5


`Tromsø`, `Málaga` and `Kraków` all came back with their accented characters intact, from three
files that were not stored the same way.

The `raise` at the end never fires while `latin-1` is in the list, because `latin-1` cannot fail.
It is there because the list is a setting, and somebody who edits it to `("utf-8",)` should get a
clear error rather than `None`.


### Stage 3: parse each row and check it

`csv.DictReader` needs something it can read lines from, and stage two produced a string.
`io.StringIO` wraps that string in the file interface `DictReader` expects, so no temporary file
is needed.

Two checks run on every row, and each has its own rejection reason:

- **A required field is empty or absent.** `DictReader` fills a short row's missing fields with
  `None`, so `row.get(field)` catches both the absent field and the empty string.
- **The reading is not a number.** `float("warm")` raises `ValueError`, and a row that fails is
  recorded rather than allowed to stop the run.

`start=2` in `enumerate` makes the recorded line number match what a text editor shows: the header
is line 1, so the first data row is line 2. A drop reason nobody can locate is not much use.


In [4]:

REQUIRED = ("date", "station", "celsius")


def clean_rows(text, source):
    """Split text into the rows that can be used and the ones that cannot."""
    kept, dropped = [], []
    for line_no, row in enumerate(csv.DictReader(io.StringIO(text)), start=2):
        missing = [field for field in REQUIRED if not row.get(field)]
        if missing:
            dropped.append({"source": source, "line": line_no,
                            "reason": f"missing {missing[0]}"})
            continue
        try:
            celsius = float(row["celsius"])
        except ValueError:
            dropped.append({"source": source, "line": line_no,
                            "reason": "celsius is not a number"})
            continue
        kept.append({"date": row["date"], "station": row["station"],
                     "celsius": celsius, "source": source})
    return kept, dropped


kept, dropped = [], []
with zipfile.ZipFile(archive) as z:
    for entry in found:
        text, _ = decode(z.read(entry["name"]))
        rows_kept, rows_dropped = clean_rows(text, entry["name"])
        kept += rows_kept
        dropped += rows_dropped

print(f"  read {len(kept) + len(dropped)} rows: kept {len(kept)}, dropped {len(dropped)}")
for row in dropped:
    print(f"    {row['source']} line {row['line']}: {row['reason']}")


  read 8 rows: kept 6, dropped 2
    2026/east.csv line 3: missing celsius
    2026/east.csv line 4: celsius is not a number


Both drops came from the same file, and the report names the file and the line for each one.
Somebody can open `2026/east.csv`, go to line 3, and see the row that was missing a reading.

Note what the pipeline did **not** do: it did not guess. A row missing its temperature is not
filled with a zero or with the column average, because either would put a number in the output
that nobody measured.


### Stage 4: the report

Everything the pipeline knows, in one dictionary: how many files, how many rows each way, why
each row went, which encodings were seen, and the per-station totals.

`Counter` turns a list of reasons into counts per reason in one call. Sorting the stations keeps
the output stable, so running the pipeline twice on the same input produces byte-identical files
and a diff shows only real changes.


In [5]:

def summarize(kept, dropped, entries):
    """Reduce the two row lists and the manifest to one report."""
    per_station = {}
    for row in kept:
        per_station.setdefault(row["station"], []).append(row["celsius"])
    return {
        "files_read": len(entries),
        "rows_read": len(kept) + len(dropped),
        "rows_kept": len(kept),
        "rows_dropped": len(dropped),
        "reasons": dict(Counter(row["reason"] for row in dropped)),
        "encodings": dict(Counter(entry["encoding"] for entry in entries)),
        "stations": {name: {"n": len(values),
                            "mean_celsius": round(sum(values) / len(values), 2)}
                     for name, values in sorted(per_station.items())},
    }


report = summarize(kept, dropped, found)
print(json.dumps(report, indent=2, ensure_ascii=False))


{
  "files_read": 4,
  "rows_read": 8,
  "rows_kept": 6,
  "rows_dropped": 2,
  "reasons": {
    "missing celsius": 1,
    "celsius is not a number": 1
  },
  "encodings": {
    "utf-8-sig": 3,
    "latin-1": 1
  },
  "stations": {
    "Bodø": {
      "n": 1,
      "mean_celsius": -2.6
    },
    "Galway": {
      "n": 1,
      "mean_celsius": 11.5
    },
    "Kraków": {
      "n": 1,
      "mean_celsius": 7.2
    },
    "Málaga": {
      "n": 2,
      "mean_celsius": 19.15
    },
    "Tromsø": {
      "n": 1,
      "mean_celsius": -4.1
    }
  }
}


`ensure_ascii=False` keeps `Tromsø` readable in the printed output. The **JSON on Disk** notebook
covered the alternative: leave it at the default and the file is still correct, but every accented
character is stored as an escape sequence and nobody can read the report without decoding it.

`rows_read` is in there so the arithmetic is visible. Six kept and two dropped make eight, and
eight is what the files contained.


### Stage 5: write the result

Two small functions, kept apart on purpose.

`to_csv_text` turns rows into CSV text and nothing else. It writes into an `io.StringIO` rather
than a file, which is the same trick stage three used for reading, in the other direction.

`write_atomic` takes finished text and puts it on disk, from the **Writing Safely** notebook:
write to a temporary file in the target's own folder, then `os.replace` it onto the target in one
operation. If the run is interrupted, the previous result is still there and intact.

Splitting them this way means the CSV formatting can be tested without touching the disk, and the
safe-write can be reused for the JSON report, which is not CSV at all.

`newline=""` is there for the reason the **CSV** notebook gave: the `csv` module writes its own
line endings, and letting the file object translate them again produces a blank line between every
row on Windows.

One limit worth naming: this builds the whole result in memory before writing any of it. For a few
thousand rows that is fine. For a result too large to hold, pass the open file to a `DictWriter`
and write rows as they are produced.


In [6]:

def to_csv_text(fieldnames, rows):
    """Render rows as CSV text, header first."""
    buffer = io.StringIO()
    writer = csv.DictWriter(buffer, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)
    return buffer.getvalue()


def write_atomic(target, text):
    """Write text beside target, then move it onto target in one operation."""
    handle, temporary = tempfile.mkstemp(dir=target.parent, suffix=".part")
    temporary = Path(temporary)
    try:
        with open(handle, "w", encoding="utf-8", newline="") as f:
            f.write(text)
        os.replace(temporary, target)
    finally:
        temporary.unlink(missing_ok=True)


out = scratch / "out"
out.mkdir()
kept.sort(key=lambda row: (row["source"], row["date"]))

write_atomic(out / "clean.csv",
             to_csv_text(["date", "station", "celsius", "source"], kept))
write_atomic(out / "dropped.csv",
             to_csv_text(["source", "line", "reason"], dropped))
write_atomic(out / "report.json", json.dumps(report, indent=2, ensure_ascii=False))

for path in sorted(out.iterdir()):
    print(f"  {path.name:<12} {path.stat().st_size:>3} bytes")


  clean.csv    263 bytes
  dropped.csv   94 bytes
  report.json  558 bytes


Three files, and the second one is the point of the exercise. `clean.csv` is what the work was
for; `dropped.csv` is what makes the result trustworthy, because anybody can open it and see
precisely what did not make it.


In [7]:

print(out.joinpath("clean.csv").read_text(encoding="utf-8"))
print(out.joinpath("dropped.csv").read_text(encoding="utf-8"))


date,station,celsius,source
2026-03-01,Kraków,7.2,2026/east.csv
2026-03-01,Tromsø,-4.1,2026/north.csv
2026-03-02,Bodø,-2.6,2026/north.csv
2026-03-01,Málaga,18.9,2026/south.csv
2026-03-02,Málaga,19.4,2026/south.csv
2026-03-01,Galway,11.5,2026/west.csv

source,line,reason
2026/east.csv,3,missing celsius
2026/east.csv,4,celsius is not a number



Each kept row carries the file it came from. That column costs almost nothing and answers the
question that always follows a surprising number: where did this one come from.


### The whole pipeline as one function

The five stages, called in order, with the intermediate lists held in local names. Nothing here is
new; it is the cells above with the printing removed.

The signature is worth noticing. It takes the archive and the output folder as arguments rather
than reading the module-level `archive` and `out`, which means it can be pointed at a different
archive without editing it, and it can be tested on a small one.


In [8]:

def run(archive_path, out_dir):
    """Read every CSV in an archive, clean it, and write one result with a report."""
    out_dir.mkdir(parents=True, exist_ok=True)
    entries, kept, dropped = manifest(archive_path), [], []

    with zipfile.ZipFile(archive_path) as z:
        for entry in entries:
            text, entry["encoding"] = decode(z.read(entry["name"]))
            rows_kept, rows_dropped = clean_rows(text, entry["name"])
            kept += rows_kept
            dropped += rows_dropped

    summary = summarize(kept, dropped, entries)
    kept.sort(key=lambda row: (row["source"], row["date"]))

    write_atomic(out_dir / "clean.csv",
                 to_csv_text(["date", "station", "celsius", "source"], kept))
    write_atomic(out_dir / "dropped.csv",
                 to_csv_text(["source", "line", "reason"], dropped))
    write_atomic(out_dir / "report.json",
                 json.dumps(summary, indent=2, ensure_ascii=False))
    return summary


summary = run(archive, scratch / "final")
print(summary["rows_kept"], "rows kept")
print(summary["rows_dropped"], "dropped:", summary["reasons"])
print("wrote:", [p.name for p in sorted(scratch.joinpath("final").iterdir())])


6 rows kept
2 dropped: {'missing celsius': 1, 'celsius is not a number': 1}
wrote: ['clean.csv', 'dropped.csv', 'report.json']


That is the output the first look promised.

### Running it twice

A pipeline you can run again without thinking about it is worth much more than one you cannot,
because the usual reason to rerun is that something went wrong and you have fixed it.

This one reads its input and never writes to it, sorts before writing so the row order does not
depend on the order the files came back in, and replaces each output rather than appending to it.
So a second run against the same archive produces the same bytes.


In [9]:

before = {p.name: p.read_bytes() for p in sorted(scratch.joinpath("final").iterdir())}
again = run(archive, scratch / "final")
after = {p.name: p.read_bytes() for p in sorted(scratch.joinpath("final").iterdir())}

print("same report: ", again == summary)
print("same bytes:  ", before == after)
print("no leftovers:", list(scratch.joinpath("final").glob("*.part")) == [])


same report:  True
same bytes:   True
no leftovers: True


The last line matters more than it looks. `write_atomic` creates a temporary file for every
output, and the `finally` removes it whether the replace happened or not. A pipeline that leaves
`.part` files behind fills a disk over a few months, and each one looks enough like a real output
to be confusing.


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/files-paths-and-formats/10-a-small-pipeline-solutions.ipynb).

**1.** Build a second archive that adds a fifth team's file, `2026/central.csv`, with two good
rows. Run `run` against it and print the report's `files_read` and `rows_kept`.


In [10]:
# your code here


**2.** Add a range check to `clean_rows`: a reading below -90 or above 60 is not a temperature
measured on this planet. Record it with the reason `celsius out of range`, and prove it fires by
running the new function on a row holding `999`.


In [11]:
# your code here


**3.** Add a `compression` key to each manifest entry, holding the stored size divided by the
original size, rounded to two places. Print the name and the ratio for every member.


In [12]:
# your code here


**4.** Reverse `ATTEMPTS` so that `latin-1` comes first, run stage two again, and print the
encoding and the second line of each file. Say in a comment which line is now wrong and why
nothing raised.


In [13]:
# your code here


**5.** Write a function `check(summary)` that returns `True` when `rows_kept` plus `rows_dropped`
equals `rows_read`, and `False` otherwise. Run it on the report from `run`, then on a copy of that
report with `rows_kept` reduced by one.


In [14]:
# your code here


**6.** Give `run` a third argument, `overwrite`, defaulting to `False`. When it is `False` and
`clean.csv` already exists in the output folder, raise `FileExistsError` before reading anything.
Show it refusing, then show it succeeding with `overwrite=True`.


In [15]:
# your code here


## Common errors

### TypeError, when you were expecting ValueError

A short row does not raise when it is read. `DictReader` fills the missing field with `None`, and
`float(None)` raises `TypeError`, which is not what `except ValueError` catches. A pipeline that
guards only against `ValueError` runs until it meets a short row and then stops on the whole
archive.


In [16]:

short = "date,station,celsius\n2026-03-02,Krak\u00f3w\n"
row = next(csv.DictReader(io.StringIO(short)))
print("what DictReader returned:", row)

float(row["celsius"])


what DictReader returned: {'date': '2026-03-02', 'station': 'Kraków', 'celsius': None}


TypeError: float() argument must be a string or a real number, not 'NoneType'

`celsius` is `None`, not `""` and not missing. That is why `clean_rows` checks for the missing
field with `row.get(field)` **before** it calls `float`: by the time `float` is reached, the
value is known to be a non-empty string. Handed the same row, `clean_rows` records it and
returns rather than raising.


In [17]:
kept_rows, dropped_rows = clean_rows(short, "short.csv")

print("kept:   ", kept_rows)
print("dropped:", dropped_rows)


kept:    []
dropped: [{'source': 'short.csv', 'line': 2, 'reason': 'missing celsius'}]


### KeyError: a file whose header is different

The fifth team names their column `temp`. Indexing the row with `row["celsius"]` raises
`KeyError` on the first row and the run ends there.


In [18]:

other = "date,station,temp\n2026-03-01,Galway,11.5\n"

for row in csv.DictReader(io.StringIO(other)):
    print(float(row["celsius"]))


KeyError: 'celsius'

`row.get("celsius")` returns `None` instead of raising, so `clean_rows` treats the row as
missing a required field, records the reason, and carries on to the rest of the archive. A
report saying that every row from one file was dropped for `missing celsius` points straight
at the header.

That is the trade the pipeline is making everywhere: a bad file should cost you that file's
rows and a line in the report, not the run.


In [19]:
row = next(csv.DictReader(io.StringIO(other)))
print("the header Python found:", list(row))

kept_rows, dropped_rows = clean_rows(other, "2026/central.csv")
print("kept:   ", kept_rows)
print("dropped:", dropped_rows)


the header Python found: ['date', 'station', 'temp']
kept:    []
dropped: [{'source': '2026/central.csv', 'line': 2, 'reason': 'missing celsius'}]


### ValueError: the writer does not know about a field you added

`DictWriter` is given a list of field names, and it raises if a row carries a key that is not in
it. This one shows up when you add a field in stage three and forget that stage five decides the
columns.


In [20]:

kept[0]["quality"] = "checked"

to_csv_text(["date", "station", "celsius", "source"], kept)


ValueError: dict contains fields not in fieldnames: 'quality'

The message names the offending key. Either add it to the field list or drop it before writing;
`extrasaction="ignore"` also silences it, at the cost of writing a file that quietly lacks a
column somebody added on purpose.


In [21]:

del kept[0]["quality"]

print(to_csv_text(["date", "station", "celsius", "source"], kept).splitlines()[1])


2026-03-01,Kraków,7.2,2026/east.csv


### The quiet one: the archive decoded, and one file is wrong

Reverse the attempt order and every file still decodes, because `latin-1` accepts any bytes at
all. Nothing raises, the pipeline reports success, and one station name is mojibake.


In [22]:
def decode_wrong(raw):
    for encoding in ("latin-1", "utf-8-sig"):
        try:
            return raw.decode(encoding), encoding
        except UnicodeDecodeError:
            continue


seen = Counter()
with zipfile.ZipFile(archive) as z:
    for entry in found:
        text, used = decode_wrong(z.read(entry["name"]))
        seen[used] += 1
        print(f"  {entry['name']:<16} {used:<10} {text.splitlines()[1]}")

print("\n  encodings the report would show:", dict(seen))


  2026/east.csv    latin-1    2026-03-01,KrakÃ³w,7.2
  2026/north.csv   latin-1    2026-03-01,TromsÃ¸,-4.1
  2026/south.csv   latin-1    2026-03-01,Málaga,18.9
  2026/west.csv    latin-1    2026-03-01,Galway,11.5

  encodings the report would show: {'latin-1': 4}


`Kraków` and `Tromsø` have become `KrakÃ³w` and `TromsÃ¸`. `Málaga` is untouched, because
`2026/south.csv` is the one file that really was latin-1. Reversing the order damaged the three
files that were fine and read correctly only the file that was not, so a spot check that happened
to land on `south.csv` would have found nothing wrong.

The report is what catches it. `encodings` reads `{'latin-1': 4}`, and four files from four
laptops all agreeing on latin-1 is not plausible. That is the value of recording how each stage
did its work rather than only what it produced.

### Cleaning up


In [23]:

shutil.rmtree(scratch)

print("scratch still there:", scratch.exists())


scratch still there: False


## Recap

- A pipeline is a sequence of stages, each with one job and testable on its own.
- Rows kept plus rows dropped equals rows read, and the pipeline reports all three.
- A malformed row is a fact about the input, not a bug. Record it and carry on.
- Read an archive's manifest with `infolist` before extracting anything, and decide what counts
  as data.
- Decode with a fixed list of attempts, and put `latin-1` last, because it never fails.
- Record which encoding worked. A report that says every file was latin-1 is telling you the
  order is wrong.
- `io.StringIO` gives `csv` a file interface over a string, for reading and for writing.
- Check for a missing field before converting, so `float` never meets `None`.
- Record the source file and line with every drop, so somebody can go and look.
- Write the result with `os.replace`, so an interrupted run leaves the previous one intact.
- Sort before writing, so a rerun on the same input produces the same bytes.


## What is next

That is the end of this guide. You can read a path, open a file with the right encoding, parse
CSV, JSON and Excel, walk a directory, look inside an archive, write without risking what is
already there, and assemble all of it into one job that reports what it did.

The **Object-Oriented Python** guide comes next. Every stage in this notebook was a function
taking arguments and returning values, and the lists were passed from one to the next by hand.
When a pipeline grows enough stages, that hand-passing is what classes exist to tidy up: the data
and the operations on it in one place.

The guides that follow it move on to the libraries, and each one reads files at some point. This
guide is the reason those chapters can say `read_csv` and move on.


---

&#8592; **Previous:** [Writing Safely](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/files-paths-and-formats/09-writing-safely.ipynb)  &nbsp;·&nbsp;  [Files, Paths and Formats Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/files-paths-and-formats.html)
